# Alexander-disease gene sets: cRAP vs CRAPome contaminant comparison

In [1]:
import re
from pathlib import Path
import pandas as pd

DATA = Path("data/GFAP cell lines_ALL mutations vs WT_for Avi.xlsx")
CONT = Path("contaminants")
MUTATIONS = ["R239H", "R239C", "R79C", "R79H", "R239G"]
ADJ_P = 0.05

In [2]:
sets = {}
for m in MUTATIONS:
    d = pd.read_excel(DATA, sheet_name=m)
    d.columns = ["gene", "log2fc", "adj_p"]
    d = d.dropna(subset=["gene", "log2fc", "adj_p"])
    d = d[d["adj_p"] < ADJ_P]
    d["gene"] = d["gene"].astype(str).str.strip()
    for _, r in d.iterrows():
        sets.setdefault(r["gene"], set()).add(f"{m}_{'up' if r['log2fc'] > 0 else 'down'}")
genes = sorted(sets)
len(genes)

362

In [3]:
cr = pd.read_excel(CONT / "crapome_human_v1.xlsx", sheet_name=0)
cc = [c for c in cr.columns if re.fullmatch(r"CC\d+", str(c))]
n_ctrl = len(cc)
freq = dict(zip(cr["geneSymbol"].astype(str), (cr[cc] > 0).sum(axis=1) / n_ctrl))
max_sc = dict(zip(cr["geneSymbol"].astype(str), cr[cc].max(axis=1)))
n_ctrl

343

In [4]:
crap = set((CONT / "crap_gene_symbols.txt").read_text().split())
published = set(pd.read_csv(CONT / "crapome_published_table.tsv", sep="\t")["gene"])
desc = dict(
    pd.read_csv(CONT / "gene_descriptions.tsv", sep="\t").fillna("").itertuples(index=False, name=None)
)

def primary_name(g):
    return desc.get(g, "").split(" (")[0]

In [5]:
report = pd.DataFrame([
    {
        "gene": g,
        "protein": primary_name(g),
        "crapome_pct": round(freq.get(g, 0) * 100, 1),
        "crapome_max_sc": int(max_sc.get(g, 0)),
        "removed_by_crap": g in crap,
        "in_crapome_published_list": g in published,
        "sets": "; ".join(sorted(sets[g])),
    }
    for g in genes
])
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 300)

## Contaminant comparison

In [6]:
(report[(report.crapome_pct >= 30) | report.removed_by_crap]
    .sort_values("crapome_pct", ascending=False)
    .reset_index(drop=True))

,gene,protein,crapome_pct,crapome_max_sc,removed_by_crap,in_crapome_published_list,sets
0,KRT9,"Keratin, type I cytoskeletal 9",81.9,699,True,True,R239C_up
1,KRT75,"Keratin, type II cytoskeletal 75",64.4,80,False,True,R239H_up; R79C_down
2,KRT18,"Keratin, type I cytoskeletal 18",53.1,311,False,True,R239H_down
3,KRT8,"Keratin, type II cytoskeletal 8",51.9,528,False,True,R239H_down; R79H_down
4,ALB,Albumin,41.4,93,True,False,R239G_down
5,MYH10,Myosin-10,33.5,205,False,False,R239H_down; R79C_up
6,FLNB,Filamin-B,31.5,121,False,True,R239H_down
7,KRT81,"Keratin, type II cuticular Hb1",9.9,143,True,False,R79C_down
